# Прогноз количества запросов на следующий час

Ноутбук оставляет только основной пайплайн: подготовка почасового ряда, признаки, модели `baseline`, `linear`, `rf`, `cat`, `arima`, таблицу метрик и два финальных графика.


In [35]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_absolute_percentage_error,
    mean_squared_error,
    r2_score,
)
from catboost import CatBoostRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings("ignore")

plt.rcParams["figure.figsize"] = (15, 6)
plt.rcParams["axes.grid"] = True


## Настройки


In [ ]:
DATA_PATH = "df_with_cat.csv"
CITY_NAME = "İstanbul"

TEST_SIZE = 0.20
VALID_SIZE = 0.20
RANDOM_STATE = 42

RF_TREES = 500
CAT_ITERATIONS = 3000


## Подготовка данных


In [37]:
from sklearn.preprocessing import OneHotEncoder

events = pd.read_csv(DATA_PATH)

events["model_response_timestamp"] = pd.to_datetime(
    events["model_response_timestamp"],
    unit="s",
)
events["date_hour"] = events["model_response_timestamp"].dt.floor("h")

events["category"] = events["category"].fillna("unknown").astype(str)

hourly = (
    events.groupby(["name", "category", "date_hour"])
    .size()
    .reset_index(name="count")
)

city_hourly = hourly.loc[
    hourly["name"] == CITY_NAME,
    ["date_hour", "category", "count"],
]

all_hours = pd.date_range(
    city_hourly["date_hour"].min(),
    city_hourly["date_hour"].max(),
    freq="h",
)

all_categories = sorted(city_hourly["category"].unique())

full_index = pd.MultiIndex.from_product(
    [all_hours, all_categories],
    names=["date_hour", "category"],
)

base_df = (
    city_hourly.set_index(["date_hour", "category"])
    .reindex(full_index, fill_value=0)
    .reset_index()
)

base_df.insert(0, "name", CITY_NAME)
base_df = base_df.sort_values(["date_hour", "category"]).reset_index(drop=True)

encoder = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

category_ohe = encoder.fit_transform(base_df[["category"]])

category_ohe_df = pd.DataFrame(
    category_ohe,
    columns=encoder.get_feature_names_out(["category"]),
    index=base_df.index,
)

base_df = pd.concat([base_df, category_ohe_df], axis=1)

base_df.head()

,name,date_hour,category,count,category_Agriculture,"category_Automotive, motorcycles and vehicles","category_Beauty, personal care and wellness","category_Construction, renovation and interior","category_Culture, arts and entertainment",category_Education and training,...,"category_Manufacturing, industry and equipment","category_Nature, parks and outdoor places",category_Other and unspecified,category_Real estate and business properties,category_Religion and community organizations,category_Retail and trade,category_Sports and active recreation,"category_Tourism, lodging and travel",category_Transportation and logistics,"category_Utilities, security and maintenance services"
0,İstanbul,2026-02-13 21:00:00,Agriculture,1,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,İstanbul,2026-02-13 21:00:00,"Automotive, motorcycles and vehicles",4,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,İstanbul,2026-02-13 21:00:00,"Beauty, personal care and wellness",1,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,İstanbul,2026-02-13 21:00:00,"Construction, renovation and interior",0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,İstanbul,2026-02-13 21:00:00,"Culture, arts and entertainment",2,0.0,0.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [38]:
base_df["count"].sum()

np.int64(17238)

In [39]:
def add_time_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values(["category", "date_hour"]).reset_index(drop=True).copy()

    result["hour"] = result["date_hour"].dt.hour
    result["day"] = result["date_hour"].dt.day
    result["month"] = result["date_hour"].dt.month
    result["day_of_week"] = result["date_hour"].dt.dayofweek
    result["is_weekend"] = result["day_of_week"].isin([5, 6]).astype(int)

    result["hour_sin"] = np.sin(2 * np.pi * result["hour"] / 24)
    result["hour_cos"] = np.cos(2 * np.pi * result["hour"] / 24)

    for day in range(7):
        result[f"day_of_week_{day}"] = (result["day_of_week"] == day).astype(int)

    return result


def add_lag_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.sort_values(["category", "date_hour"]).reset_index(drop=True).copy()

    group = result.groupby("category")["count"]

    result["lag_1"] = group.shift(1)
    result["lag_2"] = group.shift(2)
    result["lag_24"] = group.shift(24)

    result["rolling_mean_3"] = group.transform(
        lambda x: x.shift(1).rolling(3).mean()
    )
    result["rolling_mean_6"] = group.transform(
        lambda x: x.shift(1).rolling(6).mean()
    )
    result["rolling_mean_24"] = group.transform(
        lambda x: x.shift(1).rolling(24).mean()
    )

    return result


def build_features(df: pd.DataFrame) -> pd.DataFrame:
    return add_lag_features(add_time_features(df))

In [40]:
base_df = base_df.sort_values(["category", "date_hour"]).reset_index(drop=True)

featured_df = build_features(base_df)

featured_df["predict_1h"] = (
    featured_df.groupby("category")["count"].shift(-1)
)
featured_df["target_hour"] = featured_df["date_hour"] + pd.Timedelta(hours=1)

excluded_dates = featured_df["month"].eq(2) & featured_df["day"].isin([14, 15, 25])

base_excluded_dates = (
    base_df["date_hour"].dt.month.eq(2)
    & base_df["date_hour"].dt.day.isin([14, 15, 25])
)

recursive_base_df = base_df.loc[~base_excluded_dates].reset_index(drop=True)
featured_df = featured_df.loc[~excluded_dates].reset_index(drop=True)
featured_df = featured_df.sort_values(["date_hour"]).reset_index(drop=True)
day_columns = [f"day_of_week_{day}" for day in range(7)]
category_columns = [
    col for col in featured_df.columns
    if col.startswith("category_")
]
feature_cols = [
    "count",
    "hour_sin",
    "hour_cos",
    "is_weekend",
    *day_columns,
    *category_columns,
]
model_df = featured_df.dropna(subset=feature_cols + ["predict_1h"]).reset_index(drop=True)
model_df[["date_hour", "target_hour", "category", "count", "predict_1h"]].head()

,date_hour,target_hour,category,count,predict_1h
0,2026-02-13 21:00:00,2026-02-13 22:00:00,Agriculture,1,1.0
1,2026-02-13 21:00:00,2026-02-13 22:00:00,"Automotive, motorcycles and vehicles",4,0.0
2,2026-02-13 21:00:00,2026-02-13 22:00:00,"Tourism, lodging and travel",1,0.0
3,2026-02-13 21:00:00,2026-02-13 22:00:00,Sports and active recreation,3,0.0
4,2026-02-13 21:00:00,2026-02-13 22:00:00,Retail and trade,2,0.0


In [41]:
model_df["count"].sum()

np.int64(15047)

## Разделение на train / validation / test


In [42]:
split_index = int(len(model_df) * (1 - TEST_SIZE))
train_df = model_df.iloc[:split_index].copy()
test_df = model_df.iloc[split_index:].copy()

valid_index = int(len(train_df) * (1 - VALID_SIZE))
cat_train_df = train_df.iloc[:valid_index].copy()
cat_valid_df = train_df.iloc[valid_index:].copy()

X_train = train_df[feature_cols]
y_train = train_df["predict_1h"]

X_cat_train = cat_train_df[feature_cols]
y_cat_train = cat_train_df["predict_1h"]
X_cat_valid = cat_valid_df[feature_cols]
y_cat_valid = cat_valid_df["predict_1h"]

X_test = test_df[feature_cols]
y_test = test_df["predict_1h"]

print(f"train: {X_train.shape}")
print(f"train: {y_train.shape}")
print(f"cat validation: {X_cat_valid.shape}")
print(f"test: {X_test.shape}")


train: (12232, 33)
train: (12232,)
cat validation: (2447, 33)
test: (3058, 33)


## Метрики


In [43]:
scores = []
preds_df = pd.DataFrame({
    "date_hour": test_df["target_hour"].values,
    "category": test_df["category"].values,
    "true": y_test.values,
})


def save_prediction(name: str, y_true, y_pred) -> None:
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)
    preds_df[name] = y_pred

    scores.append({
        "model": name,
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred),
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
    })
def save_score(name: str, y_true, y_pred) -> None:
    y_pred = np.clip(np.asarray(y_pred, dtype=float), 0, None)

    print(f'''
        "model": {name},
        "MAE": {mean_absolute_error(y_true, y_pred)},
        "RMSE": {np.sqrt(mean_squared_error(y_true, y_pred))},
        "R2": {r2_score(y_true, y_pred)},
        "MAPE": {mean_absolute_percentage_error(y_true, y_pred)},
    ''')


## Baseline

Наивный прогноз: считаем, что в следующий час будет столько же запросов, сколько в текущий.


In [44]:
y_pred_baseline = X_test["count"].values
save_prediction("baseline", y_test, y_pred_baseline)


## Linear Regression


In [45]:
linear_model = LinearRegression()
linear_model.fit(X_train, y_train)

y_pred_linear = linear_model.predict(X_test)
y_pred_linear_train = linear_model.predict(X_train)
save_score("linear", y_test, y_pred_linear)
save_score("linear_train", y_train, y_pred_linear_train)
save_prediction("linear", y_test, y_pred_linear)



        "model": linear,
        "MAE": 0.821386782619359,
        "RMSE": 1.2563401566666244,
        "R2": 0.32481837769645794,
        "MAPE": 1604153666987639.5,
    

        "model": linear_train,
        "MAE": 0.8518989318488391,
        "RMSE": 1.3272395822667344,
        "R2": 0.398561867124548,
        "MAPE": 1458862805429514.0,
    


## Random Forest


In [46]:
rf_model = RandomForestRegressor(
    n_estimators=RF_TREES,
    max_depth=20,
    min_samples_split=20,
    min_samples_leaf=5,
    max_features="sqrt",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
save_score("rf", y_test, y_pred_rf)
save_prediction("rf", y_test, y_pred_rf)



        "model": rf,
        "MAE": 0.8063157416915989,
        "RMSE": 1.2107923852582936,
        "R2": 0.3728874503708396,
        "MAPE": 1560462469457841.2,
    


## CatBoost


In [47]:
cat_model = CatBoostRegressor(
    iterations=CAT_ITERATIONS,
    learning_rate=0.01,
    depth=8,
    l2_leaf_reg=3,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=RANDOM_STATE,
    early_stopping_rounds=100,
    verbose=200,
)
cat_model.fit(
    X_cat_train,
    y_cat_train,
    eval_set=(X_cat_valid, y_cat_valid),
    use_best_model=True,
)

y_pred_cat = cat_model.predict(X_test)
save_prediction("cat", y_test, y_pred_cat)


0:	learn: 1.6974416	test: 1.7344787	best: 1.7344787 (0)	total: 5.79ms	remaining: 17.4s
200:	learn: 1.2833773	test: 1.3730002	best: 1.3730002 (200)	total: 556ms	remaining: 7.75s
400:	learn: 1.2267097	test: 1.3412610	best: 1.3412610 (400)	total: 1.06s	remaining: 6.9s
600:	learn: 1.2000247	test: 1.3327178	best: 1.3327178 (600)	total: 1.59s	remaining: 6.33s
800:	learn: 1.1756963	test: 1.3286857	best: 1.3285686 (799)	total: 2.1s	remaining: 5.76s
1000:	learn: 1.1525678	test: 1.3272716	best: 1.3271629 (978)	total: 2.62s	remaining: 5.22s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 1.327162931
bestIteration = 978

Shrink model to first 979 iterations.


## Сводная таблица


In [48]:
scores_df = pd.DataFrame(scores).sort_values("RMSE").reset_index(drop=True)
display(scores_df)


,model,MAE,RMSE,R2,MAPE
0,rf,0.806316,1.210792,0.372887,1.560462e+15
1,cat,0.809143,1.222729,0.360462,1.560678e+15
2,linear,0.821387,1.256340,0.324818,1.604154e+15
3,baseline,0.907456,1.639222,-0.149428,1.235618e+15


In [55]:
scores_df = pd.DataFrame(scores).sort_values("RMSE").reset_index(drop=True)
display(scores_df)

model_cols = [
    col for col in preds_df.columns
    if col not in ["date_hour", "category", "true"]
]

category_scores = []

for category, category_df in preds_df.groupby("category"):
    category_true_sum = category_df["true"].sum()
    category_true_share = category_true_sum / preds_df["true"].sum()

    for model in ['rf']:
        y_test_category = category_df["true"].values
        y_pred_category = category_df[model].values

        category_scores.append({
            "category": category,
            "model": model,
            # "n": len(category_df),
            # "true_sum": category_true_sum,
            # "pred_sum": y_pred_category.sum(),
            # "true_share": category_true_share,
            # "pred_share": y_pred_category.sum() / preds_df[model].sum(),
            "MAE": mean_absolute_error(y_test_category, y_pred_category),
            "RMSE": np.sqrt(mean_squared_error(y_test_category, y_pred_category)),
            "R2": r2_score(y_test_category, y_pred_category) if len(category_df) > 1 else np.nan,
            "MAPE": mean_absolute_percentage_error(y_test_category, y_pred_category),
        })

category_scores_df = (
    pd.DataFrame(category_scores)
    .sort_values(["category", "RMSE"])
    .reset_index(drop=True)
)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)
display(category_scores_df)
print(category_scores_df["MAE"].mean())

,model,MAE,RMSE,R2,MAPE
0,rf,0.806316,1.210792,0.372887,1.560462e+15
1,cat,0.809143,1.222729,0.360462,1.560678e+15
2,linear,0.821387,1.256340,0.324818,1.604154e+15
3,baseline,0.907456,1.639222,-0.149428,1.235618e+15


,category,model,MAE,RMSE,R2,MAPE
0,Agriculture,rf,0.625941,0.730302,0.007066,1.728674e+15
1,"Automotive, motorcycles and vehicles",rf,0.833482,1.096282,0.057834,2.140192e+15
2,"Beauty, personal care and wellness",rf,0.469481,0.606295,0.004169,1.121676e+15
3,"Construction, renovation and interior",rf,0.492323,0.666384,-0.051964,1.407081e+15
4,"Culture, arts and entertainment",rf,0.600100,0.829264,0.049189,1.215912e+15
5,Education and training,rf,0.620615,1.002785,-0.014346,1.759767e+15
6,Finance and professional services,rf,0.162960,0.280281,-0.144650,4.983659e+14
7,Food and beverages,rf,1.911018,2.542137,0.308164,1.915116e+15
8,Government and municipal services,rf,1.090580,1.344671,0.071024,2.011046e+15
9,Healthcare and medical services,rf,0.911943,1.338433,-0.050048,2.113265e+15


0.8063157416915988
